# 확률 분포 및 손실 함수(MSE, Cross-Entropy)의 MLE 기반 유도

본 노트북에서는 다음을 수행합니다.
1. 정규분포 $N(0, 1)$과 $N(2, 0.5)$의 확률밀도함수(PDF) 비교 시각화
2. 베르누이 분포 $B(0.3)$과 $B(0.7)$의 확률질량함수(PMF) 비교 시각화
3. 수치 안정적 Softmax 구현 및 확률 합 검증 ($|\sum p_i - 1.0| < 10^{-6}$)
4. **MSE 최소화 = 정규분포 가정 하의 최대우도추정(MLE)** 수식 유도
5. **Cross-Entropy 최소화 = 베르누이/카테고리 분포 가정 하의 MLE** 수식 유도


In [1]:
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams["axes.unicode_minus"] = False

OUTPUT_DIR = Path("outputs") if Path("src").exists() else Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def normal_pdf(x: np.ndarray, mu: float, sigma: float) -> np.ndarray:
    """정규분포 N(mu, sigma^2)의 확률밀도함수(PDF)를 계산합니다."""
    coeff = 1.0 / (np.sqrt(2.0 * np.pi) * sigma)
    exponent = -((x - mu) ** 2) / (2.0 * (sigma ** 2))
    return coeff * np.exp(exponent)

# 1. 정규분포 N(0, 1)과 N(2, 0.5) PDF 비교 시각화
x_vals = np.linspace(-4.0, 5.0, 400)
pdf_1 = normal_pdf(x_vals, mu=0.0, sigma=1.0)
pdf_2 = normal_pdf(x_vals, mu=2.0, sigma=0.5)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x_vals, pdf_1, label="N(0, 1): mu=0, sigma=1.0", linewidth=2.2, color="#2980b9")
ax.plot(x_vals, pdf_2, label="N(2, 0.5): mu=2, sigma=0.5", linewidth=2.2, color="#e74c3c", linestyle="--")
ax.fill_between(x_vals, pdf_1, alpha=0.15, color="#2980b9")
ax.fill_between(x_vals, pdf_2, alpha=0.15, color="#e74c3c")
ax.set_title("Normal Distribution PDF Comparison: N(0, 1) vs N(2, 0.5)")
ax.set_xlabel("x")
ax.set_ylabel("Density f(x)")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "prob_normal_pdf.png", dpi=150)
plt.show()


In [2]:
def bernoulli_pmf(k: np.ndarray, p: float) -> np.ndarray:
    """베르누이 분포 B(p)의 확률질량함수(PMF): p^k * (1-p)^(1-k)."""
    return (p ** k) * ((1.0 - p) ** (1.0 - k))

# 2. 베르누이 분포 B(0.3)과 B(0.7) PMF 막대그래프 비교
k_vals = np.array([0, 1])
pmf_03 = bernoulli_pmf(k_vals, 0.3)
pmf_07 = bernoulli_pmf(k_vals, 0.7)

fig, ax = plt.subplots(figsize=(7, 4.5))
width = 0.3
ax.bar(k_vals - width / 2, pmf_03, width=width, label="B(0.3)", color="#3498db", alpha=0.85)
ax.bar(k_vals + width / 2, pmf_07, width=width, label="B(0.7)", color="#e67e22", alpha=0.85)
ax.set_xticks(k_vals)
ax.set_xticklabels(["0 (Failure)", "1 (Success)"])
ax.set_ylim(0.0, 0.9)
ax.set_title("Bernoulli Distribution PMF Comparison: B(0.3) vs B(0.7)")
ax.set_ylabel("Probability P(X = k)")
ax.grid(True, axis="y", linestyle=":", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "prob_bernoulli_pmf.png", dpi=150)
plt.show()


In [3]:
def softmax(logits: np.ndarray) -> np.ndarray:
    """수치 안정성을 고려한 Softmax 함수: exp(z - max(z)) / sum(exp(z - max(z)))."""
    z = np.asarray(logits, dtype=np.float64)
    shifted = z - np.max(z)
    exp_z = np.exp(shifted)
    return exp_z / np.sum(exp_z)

sample_logits = np.array([2.0, 1.0, 0.1, -0.5, 3.2])
probs = softmax(sample_logits)
prob_sum = float(np.sum(probs))
diff_from_one = abs(prob_sum - 1.0)

print(f"입력 Logits : {sample_logits}")
print(f"Softmax 확률: {np.round(probs, 6)}")
print(f"확률 총합   : {prob_sum:.12f} (1.0과의 차이: {diff_from_one:.2e})")
assert diff_from_one < 1e-6, "Softmax 출력 합이 1.0과 1e-6 이상 차이납니다."
print("[검증 성공] Softmax 출력 벡터 합이 1.0과의 차이 1e-6 이내입니다.")


입력 Logits : [ 2.   1.   0.1 -0.5  3.2]
Softmax 확률: [0.203266 0.074778 0.030402 0.016685 0.674869]
확률 총합   : 1.000000000000 (1.0과의 차이: 0.00e+00)
[검증 성공] Softmax 출력 벡터 합이 1.0과의 차이 1e-6 이내입니다.


## 4. MSE 최소화 = 정규분포 가정 하의 MLE 유도

회귀 문제에서 타깃 $y_i$가 모델 예측값 $\hat{y}_i = f_\theta(x_i)$를 평균으로 하고 분산이 $\sigma^2$인 정규분포(가우시안 노이즈 $\epsilon_i \sim N(0, \sigma^2)$)를 따른다고 가정합니다:
$$p(y_i \mid x_i; \theta) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_i - \hat{y}_i)^2}{2\sigma^2}\right)$$

$N$개의 독립 항등 분포(i.i.d.) 데이터에 대한 **우도 함수(Likelihood)**는 다음과 같습니다:
$$\mathcal{L}(\theta) = \prod_{i=1}^{N} p(y_i \mid x_i; \theta)$$

양변에 로그를 취하고 부호를 뒤집어 **음의 로그 우도(Negative Log-Likelihood, NLL)**를 구하면:
$$-\log \mathcal{L}(\theta) = \sum_{i=1}^{N} \left[ \frac{1}{2}\log(2\pi\sigma^2) + \frac{(y_i - \hat{y}_i)^2}{2\sigma^2} \right]$$

여기서 $\sigma$는 파라미터 $\theta$와 무관한 상수이므로, $-\log \mathcal{L}(\theta)$를 최소화하는 $\theta_{\text{MLE}}$는 결국 오차 제곱합을 최소화하는 값과 같습니다:
$$\theta_{\text{MLE}} = \arg\min_{\theta} \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2 = \arg\min_{\theta} \text{MSE}(\theta)$$

즉, **MSE를 최소화하는 것은 데이터가 정규분포 오차를 가진다고 가정할 때의 최대우도추정(MLE)과 수학적으로 동일**합니다.


## 5. Cross-Entropy 최소화 = 베르누이/카테고리 분포 가정 하의 MLE 유도

### 5.1 이진 분류 (베르누이 분포)
정답 $y_i \in \{0, 1\}$이 확률 $\hat{y}_i = P(y_i=1 \mid x_i; \theta)$의 베르누이 분포를 따른다고 가정합니다:
$$p(y_i \mid x_i; \theta) = \hat{y}_i^{y_i} (1 - \hat{y}_i)^{1 - y_i}$$

$N$개 샘플의 음의 로그 우도(NLL)를 구하면:
$$-\log \mathcal{L}(\theta) = -\sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i) \right]$$

양변을 $N$으로 나누면 정확히 **Binary Cross-Entropy (BCE)** 손실 함수가 됩니다.

### 5.2 다중 클래스 분류 (카테고리 분포)
$C$개의 클래스 중 원-핫 인코딩된 정답 $y_i \in \{0, 1\}^C$가 모델의 Softmax 출력 확률 $\hat{y}_i = (\hat{y}_{i,1}, \dots, \hat{y}_{i,C})$를 모수로 하는 카테고리 분포를 따른다고 가정합니다:
$$p(y_i \mid x_i; \theta) = \prod_{c=1}^{C} \hat{y}_{i,c}^{y_{i,c}}$$

음의 로그 우도(NLL)를 취하면:
$$-\log \mathcal{L}(\theta) = -\sum_{i=1}^{N} \sum_{c=1}^{C} y_{i,c} \log(\hat{y}_{i,c})$$

이는 **Categorical Cross-Entropy** 손실 함수와 완전히 일치합니다. 따라서 **Cross-Entropy 손실 함수를 최소화하는 것은 베르누이/카테고리 분포 가정 하의 MLE를 수행하는 것과 동일**합니다.
